In [60]:
!pip install opendatasets --quiet
import opendatasets as od
od.download("https://www.kaggle.com/datasets/mssmartypants/rice-type-classification")


Skipping, found downloaded files in "./rice-type-classification" (use force=True to force download)


In [61]:
import torch
import torch.nn as np
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [62]:
data_df = pd.read_csv("/content/rice-type-classification/riceClassification.csv")
data_df.head(5)

,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,1,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,4,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,5,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [63]:
data_df.dropna(inplace = True)
data_df.drop(['id'], axis = 1, inplace = True)

In [64]:
data_df.head(4)

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1


In [65]:
print(data_df.shape)

(18185, 11)


In [66]:
print(data_df["Class"].unique())

[1 0]


In [67]:
print(data_df["Class"].value_counts())

Class
1    9985
0    8200
Name: count, dtype: int64


In [68]:
original_df = data_df.copy()

for column in data_df.columns:
  data_df[column] = data_df[column]/data_df[column].abs().max()

data_df.head()


,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,0.444368,0.503404,0.775435,0.744658,0.424873,0.666610,0.741661,0.537029,0.844997,0.368316,1.0
1,0.281293,0.407681,0.622653,0.750489,0.273892,0.530370,0.804230,0.409661,0.919215,0.371471,1.0
2,0.298531,0.416421,0.630442,0.756341,0.284520,0.546380,0.856278,0.412994,0.959862,0.374747,1.0
3,0.300979,0.420463,0.629049,0.764024,0.286791,0.548616,0.883772,0.414262,0.961818,0.379222,1.0
4,0.361704,0.464626,0.682901,0.775033,0.345385,0.601418,0.867808,0.452954,0.966836,0.386007,1.0


In [69]:
X = np.array(data_df.iloc[:, :-1])
y = np.array(data_df.iloc[:, -1])

In [70]:
X

array([[0.44436827, 0.50340371, 0.77543522, ..., 0.5370287 , 0.844997  ,
        0.36831616],
       [0.28129285, 0.40768133, 0.62265269, ..., 0.40966075, 0.91921498,
        0.37147093],
       [0.29853085, 0.41642141, 0.63044229, ..., 0.41299402, 0.95986205,
        0.37474651],
       ...,
       [0.62340842, 0.84480035, 0.64091576, ..., 0.67304935, 0.75472018,
        0.74783024],
       [0.58374143, 0.8263563 , 0.62355087, ..., 0.67524793, 0.70210346,
        0.75187447],
       [0.60078355, 0.83554818, 0.62495614, ..., 0.6658912 , 0.74305096,
        0.7585284 ]])

In [71]:
y

array([1., 1., 1., ..., 0., 0., 0.])

In [72]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.4)

In [73]:
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size = 0.7)

In [74]:
print(X_train.shape)
print(X_test.shape)
print(X_val.shape)

(10911, 10)
(2182, 10)
(5092, 10)


In [75]:
class dataset(Dataset):
  def __init__(self, X, y):
    self.X = torch.tensor(X, dtype = torch.float32).to(device)
    self.y = torch.tensor(y, dtype = torch.float32).to(device)

  def __len__(self):
    return len(self.X)

  def __getitem__(self, index):
    return self.X[index], self.y[index]




In [76]:
training_data = dataset(X_train, y_train)
validation_data = dataset(X_val, y_val)
testing_data = dataset(X_test, y_test)

In [77]:
train_dataloader = DataLoader(training_data, batch_size = 13, shuffle = True)
validation_dataloader = DataLoader(validation_data, batch_size = 13, shuffle = True)
testing_dataloader = DataLoader(testing_data, batch_size = 13, shuffle = True)

In [78]:
for x, y in train_dataloader:
  print(x)
  print("======")
  print(y)
  break

tensor([[0.6419, 0.8061, 0.6895, 0.9545, 0.6054, 0.8012, 0.7187, 0.6632, 0.8003,
         0.6633],
        [0.5929, 0.7998, 0.6551, 0.9614, 0.5702, 0.7700, 0.6140, 0.6658, 0.7335,
         0.6927],
        [0.4217, 0.6864, 0.5432, 0.9664, 0.4044, 0.6494, 0.5557, 0.5556, 0.7493,
         0.7170],
        [0.5960, 0.5802, 0.9115, 0.7306, 0.5609, 0.7720, 0.7061, 0.5794, 0.9737,
         0.3611],
        [0.7074, 0.8717, 0.7120, 0.9618, 0.6713, 0.8411, 0.6158, 0.7095, 0.7706,
         0.6947],
        [0.7638, 0.8004, 0.8292, 0.9148, 0.7241, 0.8739, 0.7134, 0.6887, 0.8831,
         0.5476],
        [0.7897, 0.8371, 0.8200, 0.9282, 0.7557, 0.8887, 0.7494, 0.7178, 0.8405,
         0.5792],
        [0.9678, 0.9010, 0.9356, 0.9142, 0.9260, 0.9838, 0.6834, 0.7904, 0.8495,
         0.5464],
        [0.7185, 0.7789, 0.7993, 0.9172, 0.6820, 0.8477, 0.7180, 0.6699, 0.8781,
         0.5529],
        [0.6608, 0.8820, 0.6543, 0.9749, 0.6260, 0.8129, 0.5246, 0.7067, 0.7256,
         0.7648],
        [0

In [79]:
import torch.nn as nn

HIDDEN_NEURONS = 14

class MyModel(nn.Module):
  def __init__(self):
    super(MyModel, self).__init__()

    self.input_layer = nn.Linear(X.shape[1], HIDDEN_NEURONS)
    self.linear = nn.Linear(HIDDEN_NEURONS, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    x = self.input_layer(x)
    x = self.linear(x)
    x = self.sigmoid(x)
    return x

model = MyModel().to(device)


In [80]:
summary(model, (X.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 14]             154
            Linear-2                    [-1, 1]              15
           Sigmoid-3                    [-1, 1]               0
Total params: 169
Trainable params: 169
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


In [88]:
labels = labels.unsqueeze(1)

In [89]:
criterion = nn.BCELoss()
optimizer = Adam(model.parameters(), lr = 1e-5)

In [114]:
total_loss_train_plot = []
total_loss_validation_plot = []
total_accuracy_train_plot = []
total_accuracy_validation_plot = []

epochs = 15
for epoch in range(epochs):
  total_accuracy_train = 0 # Incrementation.
  total_loss_train = 0
  total_accuracy_validation = 0
  total_loss_validation = 0

  for data in train_dataloader:
    inputs, labels = data

    prediction = model(inputs)

    labels = labels.unsqueeze(1)

    batch_loss = criterion(prediction, labels)

    total_loss_train += batch_loss.item()

    acc = ((prediction).round() == labels).sum().item()

    total_accuracy_train += acc

    batch_loss.backward()
    optimizer.step()
    optimizer.zero_grad()

  with torch.no_grad():
    for data in validation_dataloader:
      inputs, labels = data

      prediction = model(inputs).squeeze(1)
      batch_loss = criterion(prediction, labels)

      total_loss_validation += batch_loss.item()
      acc = ((prediction).round() == labels).sum().item()

      total_accuracy_validation += acc

  total_loss_train_plot.append(round(total_loss_train/1000, 4))
  total_loss_validation_plot.append(round(total_loss_validation/1000, 4))

  total_accuracy_train_plot.append(round(total_accuracy_train/training_data.__len__() * 100, 4))
  total_accuracy_validation_plot.append(round(total_accuracy_validation/validation_data.__len__() * 100, 4))

  print(f'''Epoch no. {epoch + 1} Train Loss: {round(total_loss_train/1000, 4)} Train Accuracy {round(total_accuracy_train/training_data.__len__() * 100, 4)}
          Validation Loss: {round(total_loss_validation/1000, 4)} Validation Accuracy: {round(total_accuracy_validation/validation_data.__len__() * 100, 4)}''')

  print("="*30)



Epoch no. 1 Train Loss: 0.4858 Train Accuracy 96.0957
          Validation Loss: 0.2262 Validation Accuracy: 95.9152
Epoch no. 2 Train Loss: 0.4822 Train Accuracy 95.9674
          Validation Loss: 0.2245 Validation Accuracy: 96.3276
Epoch no. 3 Train Loss: 0.4786 Train Accuracy 96.4806
          Validation Loss: 0.2228 Validation Accuracy: 96.3865
Epoch no. 4 Train Loss: 0.4749 Train Accuracy 96.4898
          Validation Loss: 0.221 Validation Accuracy: 96.5829
Epoch no. 5 Train Loss: 0.4711 Train Accuracy 96.8381
          Validation Loss: 0.2192 Validation Accuracy: 96.7007
Epoch no. 6 Train Loss: 0.4672 Train Accuracy 96.8197
          Validation Loss: 0.2174 Validation Accuracy: 96.956
Epoch no. 7 Train Loss: 0.4634 Train Accuracy 96.8839
          Validation Loss: 0.2155 Validation Accuracy: 96.9953
Epoch no. 8 Train Loss: 0.4593 Train Accuracy 96.9755
          Validation Loss: 0.2136 Validation Accuracy: 97.1917
Epoch no. 9 Train Loss: 0.4552 Train Accuracy 97.113
          Val

In [103]:

print(acc)

5
